In [ ]:
!pip install -q transformers[torch] datasets pandas scikit-learn

In [ ]:
!pip -q install huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import pandas as pd
import json

with open('synthetic_dataset.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)
# Define the number of samples per topic in the test set
test_samples_per_topic = 8

# Initialize empty dataframes for train and test
train_df = pd.DataFrame()
test_df = pd.DataFrame()

# Group by topic_label and sample for the test set
for topic, topic_df in df.groupby('topic_label'):
    if len(topic_df) > test_samples_per_topic:
        test_subset = topic_df.sample(n=test_samples_per_topic, random_state=42)
        train_subset = topic_df.drop(test_subset.index)
    else:
        # If a topic has less than or equal to the desired test samples,
        # put all of them in the test set and none in the train set for this topic
        test_subset = topic_df
        train_subset = pd.DataFrame() # Empty dataframe for this topic's train subset

    train_df = pd.concat([train_df, train_subset])
    test_df = pd.concat([test_df, test_subset])

# Shuffle the resulting dataframes
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Create the DatasetDict
raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'test': Dataset.from_pandas(test_df)
})

print("Train and test datasets created with stratified sampling.")
print(raw_datasets)

Train and test datasets created with stratified sampling.
DatasetDict({
    train: Dataset({
        features: ['history', 'current_query', 'expanded_query', 'topic_label'],
        num_rows: 1074
    })
    test: Dataset({
        features: ['history', 'current_query', 'expanded_query', 'topic_label'],
        num_rows: 160
    })
})


In [ ]:
df.isna().sum()

,0
history,0
current_query,0
expanded_query,0
topic_label,0


In [ ]:
print("Topic label distribution in the training set:")
display(train_df['topic_label'].value_counts())

print("\nTopic label distribution in the testing set:")
display(test_df['topic_label'].value_counts())

Topic label distribution in the training set:


,count
topic_label,
International Relations - Global Groupings,56
History - Medieval India,55
General Query - Meta,55
Environment & Ecology - Indian Context,55
History - Modern India,55
Science & Technology - Core Concepts,55
Polity & Governance - Comparative,55
Science & Technology - IT & Communication,55
History - World,55



Topic label distribution in the testing set:


,count
topic_label,
International Relations - Bilateral,8
Science & Technology - Core Concepts,8
General Query - Meta,8
History - Modern India,8
Economy - India,8
History - World,8
Economy - Global,8
History - Medieval India,8
Current Affairs - National,8


## Prepare Labels for Topic Tagging Model


In [ ]:
# Get all unique topic labels from the entire dataset
all_labels = df['topic_label'].unique().tolist()
label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for i, label in enumerate(all_labels)}

num_labels = len(all_labels)
print(f"Number of unique topic labels: {num_labels}")
id2label

Number of unique topic labels: 20


{0: 'History - Ancient India',
 1: 'History - Medieval India',
 2: 'History - Modern India',
 3: 'History - World',
 4: 'Polity & Governance - India',
 5: 'Polity & Governance - Comparative',
 6: 'Geography - India',
 7: 'Geography - World',
 8: 'Economy - India',
 9: 'Economy - Global',
 10: 'International Relations - Bilateral',
 11: 'International Relations - Global Groupings',
 12: 'Science & Technology - Core Concepts',
 13: 'Science & Technology - Space & Defense',
 14: 'General Query - Meta',
 15: 'Science & Technology - IT & Communication',
 16: 'Environment & Ecology - Core Concepts',
 17: 'Environment & Ecology - Indian Context',
 18: 'Current Affairs - National',
 19: 'Current Affairs - International'}

## Query Expansion Model (T5)

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments, DataCollatorForSeq2Seq

model_name_t5 = "t5-small"
tokenizer_t5 = T5Tokenizer.from_pretrained(model_name_t5)
model_t5 = T5ForConditionalGeneration.from_pretrained(model_name_t5)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

#### Preprocessing Function for T5: format your history and current_query into a single input string

In [ ]:
max_input_length = 512
max_target_length = 256

def preprocess_function_t5(examples):

    inputs = [
        f"history: {h} query: {q}" if h else f"query: {q}"
        for h, q in zip(examples["history"], examples["current_query"])
    ]

    # Tokenize inputs
    model_inputs = tokenizer_t5(inputs, max_length=max_input_length, truncation=True)

    # Tokenize targets (expanded_query)
    with tokenizer_t5.as_target_tokenizer():
        labels = tokenizer_t5(examples["expanded_query"], max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_datasets_t5 = raw_datasets.map(
    preprocess_function_t5, batched=True, remove_columns=df.columns.tolist()
)

data_collator_t5 = DataCollatorForSeq2Seq(tokenizer_t5, model=model_t5)

print(tokenized_datasets_t5)

Map:   0%|          | 0/1074 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1074
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 160
    })
})


#### Train T5 Model:

In [ ]:
training_args_t5 = TrainingArguments(
    output_dir="./t5_expansion_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=8,
    fp16=True,
    logging_dir='./t5_logs',
    logging_strategy="epoch",
    load_best_model_at_end=True, # Load the best model based on validation loss
    metric_for_best_model="eval_loss",
    report_to="none",
    )

trainer_t5 = Trainer(
    model=model_t5,
    args=training_args_t5,
    train_dataset=tokenized_datasets_t5["train"],
    eval_dataset=tokenized_datasets_t5["test"],
    tokenizer=tokenizer_t5,
    data_collator=data_collator_t5,
)

trainer_t5.train()

/tmp/ipython-input-3604327099.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_t5 = Trainer(


Epoch,Training Loss,Validation Loss
1,1.716600,0.990548
2,1.366500,0.892339
3,1.262400,0.841631
4,1.209900,0.820993
5,1.174800,0.804161
6,1.145500,0.794914
7,1.138600,0.789686
8,1.127100,0.788105


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1080, training_loss=1.267669748376917, metrics={'train_runtime': 182.8444, 'train_samples_per_second': 46.991, 'train_steps_per_second': 5.907, 'total_flos': 666597494489088.0, 'train_loss': 1.267669748376917, 'epoch': 8.0})

In [ ]:
trainer_t5.save_model("./final_t5_expansion_model")
tokenizer_t5.save_pretrained("./final_t5_expansion_model")
print("T5 model training complete and saved.")

T5 model training complete and saved.


In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, pipeline
from tqdm.auto import tqdm
import pandas as pd
tqdm.pandas()

model_path = "./final_t5_expansion_model"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

expansion_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0
)

def generate_expansion(row):
    input_text = f"history: {row['history']} query: {row['current_query']}"
    result = expansion_pipeline(input_text, max_new_tokens=256, num_beams=4)
    return result[0]['generated_text']

test_df['expanded_query_t5'] = test_df.progress_apply(generate_expansion, axis=1)
train_df['expanded_query_t5'] = train_df.progress_apply(generate_expansion, axis=1)

Device set to use cuda:0


  0%|          | 0/160 [00:00<?, ?it/s]

  0%|          | 0/1074 [00:00<?, ?it/s]

In [ ]:
display(train_df[[
    'history',
    'current_query',
    'expanded_query',
    'expanded_query_t5'
]])

,history,current_query,expanded_query,expanded_query_t5
0,user: Tell me about the Mughal Empire. | bot: ...,what led to its decline,What factors led to the decline of the Mughal ...,What led to the decline of the Mughal Empire?
1,user: Describe the monsoon mechanism in India....,and its impact on agriculture?,What is the impact of the Indian monsoon on ag...,What is the impact of the monsoon mechanism on...
2,user: What is 'Project Tiger'? | bot: Project ...,what is the latest tiger census report,What are the findings of the latest tiger cens...,What is the latest tiger census report?
3,user: Tell me about the major schools of ancie...,explain the samkhya one,Explain the Samkhya school of ancient Indian p...,Explain the samkhya one of the major orthodox ...
4,user: I want to know about the structure of th...,thank you for the information,thank you for the information,Thank you for the information.
...,...,...,...,...
1069,user: Explain the law of demand. | bot: The la...,never mind,never mind,Never mind the law of demand
1070,user: What are 'ocean currents'? | bot: Ocean ...,how does that affect global weather,How does El Niño affect global weather patterns?,How does El Nio affect global weather?
1071,user: What are the Bretton Woods institutions?...,how are their roles different today,How are the roles of the World Bank and the IM...,How are the roles of the Bretton Woods institu...
1072,user: What is a 'satellite'? | bot: A satellit...,what is a 'geostationary' orbit,What is a 'geostationary' orbit?,What is a 'geostationary' orbit?


In [ ]:
display(test_df[[
    'history',
    'current_query',
    'expanded_query',
    'expanded_query_t5'
]])

,history,current_query,expanded_query,expanded_query_t5
0,user: What is India's 'Neighborhood First' pol...,how has it played out with nepal,How has India's 'Neighborhood First' policy pl...,How has India's 'Neighborhood First' policy pl...
1,user: What is India's relationship with the Gu...,what is the 'i2u2' grouping,What is the 'I2U2' grouping and which Gulf cou...,What is the 'i2u2' grouping?
2,user: What is an 'antibiotic'? | bot: An antib...,what is 'antibiotic resistance',What is 'antibiotic resistance'?,What is 'antibiotic resistance'?
3,"user: Hello, I'd like to start a research on c...","Hello, I'd like to start a research on cryptoc...","Hello, I'd like to start a research on cryptoc...","Hello, I'd like to start a research on cryptoc..."
4,user: Tell me about the Indian National Army (...,what happened to it after the war,What happened to the Indian National Army afte...,What happened to the Indian National Army (INA...
...,...,...,...,...
155,user: Explain the theory of Continental Drift....,who proposed it,Who first proposed the theory of Continental D...,Who proposed the theory of Continental Drift?
156,user: Tell me about India's 'Neighborhood Firs...,what are the challenges to this policy,What are the main challenges to India's 'Neigh...,What are the challenges to India's 'Neighborho...
157,"user: What's the update on the 'One Nation, On...",what were the main challenges in its implement...,What were the main challenges in implementing ...,What were the main challenges in the implement...
158,user: What led to the Battle of Plassey in 175...,what was its long term impact on india,What was the long-term impact of the Battle of...,What was the long term impact of the Battle of...


## Topic Tagging Model (DistilBERT)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

model_name_bert = "distilbert-base-uncased"
tokenizer_bert = AutoTokenizer.from_pretrained(model_name_bert)
model_bert = AutoModelForSequenceClassification.from_pretrained(
    model_name_bert, num_labels=num_labels, id2label=id2label, label2id=label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Preprocessing Function for DistilBERT: This will tokenize the expanded_query and map the topic_label to its numerical ID.

In [ ]:
max_length_bert = 256

def preprocess_function_bert(examples):
    tokenized_inputs = tokenizer_bert(
        examples["expanded_query_t5"], # Use expanded_query_t5
        truncation=True,
        max_length=max_length_bert
    )
    tokenized_inputs["labels"] = [label2id[l] for l in examples["topic_label"]]
    return tokenized_inputs

In [ ]:
raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'test': Dataset.from_pandas(test_df)
})

columns_to_remove_bert = ['history', 'current_query', 'expanded_query'] # Remove expanded_query as we're using expanded_query_t5

tokenized_datasets_bert = raw_datasets.map(
    preprocess_function_bert, batched=True, remove_columns=columns_to_remove_bert
)
print(tokenized_datasets_bert)

Map:   0%|          | 0/1074 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['topic_label', 'expanded_query_t5', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1074
    })
    test: Dataset({
        features: ['topic_label', 'expanded_query_t5', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 160
    })
})


#### Define Metrics for Evaluation

In [ ]:
def compute_metrics(p):
    predictions = p.predictions.argmax(axis=1)
    labels = p.label_ids

    # Calculate accuracy
    accuracy = accuracy_score(labels, predictions)

    # Calculate F1-score (weighted to handle potential class imbalance)
    # precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='macro', zero_division=0)

    return {"accuracy": accuracy, "f1": f1}

####Train DistilBERT Model

In [ ]:
training_args_bert = TrainingArguments(
    output_dir="./bert_topic_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8, # Adjust epochs if needed
    weight_decay=0.01,
    fp16=True,
    logging_dir='./bert_logs',
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1", # Evaluate based on F1-score for classification
    greater_is_better=True,
    report_to="none"
)

trainer_bert = Trainer(
    model=model_bert,
    args=training_args_bert,
    train_dataset=tokenized_datasets_bert["train"],
    eval_dataset=tokenized_datasets_bert["test"],
    tokenizer=tokenizer_bert,
    compute_metrics=compute_metrics,
)

trainer_bert.train()

/tmp/ipython-input-4180348614.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_bert = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.854000,2.580408,0.487500,0.464485
2,2.298100,2.012781,0.643750,0.624524
3,1.773000,1.614511,0.687500,0.678350
4,1.380700,1.344647,0.775000,0.773351
5,1.114600,1.194811,0.768750,0.767825
6,0.924900,1.098461,0.781250,0.776348
7,0.810400,1.041631,0.793750,0.790771
8,0.749900,1.019461,0.768750,0.764826


TrainOutput(global_step=544, training_loss=1.4882060639998491, metrics={'train_runtime': 143.0249, 'train_samples_per_second': 60.073, 'train_steps_per_second': 3.804, 'total_flos': 51768934636320.0, 'train_loss': 1.4882060639998491, 'epoch': 8.0})

In [ ]:
# Save the fine-tuned BERT model and tokenizer
trainer_bert.save_model("./final_bert_topic_model")
tokenizer_bert.save_pretrained("./final_bert_topic_model")
print("DistilBERT model training complete and saved.")

DistilBERT model training complete and saved.


## Integrated Inference and Evaluation (Post-Training)

In [ ]:

from transformers import pipeline

# Load T5 model and tokenizer
t5_tokenizer_inf = T5Tokenizer.from_pretrained("./final_t5_expansion_model")
t5_model_inf = T5ForConditionalGeneration.from_pretrained("./final_t5_expansion_model")

# Load BERT model and tokenizer
bert_tokenizer_inf = AutoTokenizer.from_pretrained("./final_bert_topic_model")
bert_model_inf = AutoModelForSequenceClassification.from_pretrained(
    "./final_bert_topic_model", id2label=id2label, label2id=label2id
)

# Create T5 pipeline for text generation
t5_pipeline = pipeline(
    "text2text-generation",
    model=t5_model_inf,
    tokenizer=t5_tokenizer_inf,
    device=0 # For GPU, use -1 for CPU
)

# Create BERT pipeline for text classification
bert_pipeline = pipeline(
    "text-classification",
    model=bert_model_inf,
    tokenizer=bert_tokenizer_inf,
    device=0 # For GPU, use -1 for CPU
)

Device set to use cuda:0
Device set to use cuda:0


In [ ]:
def get_full_prediction(conversation_history_str, current_user_query_str):
    # 1. Query Expansion (T5)
    t5_input = f"history: {conversation_history_str} query: {current_user_query_str}" if conversation_history_str else f"query: {current_user_query_str}"

    # Generate expanded query, ensuring we get the text output
    expanded_query_result = t5_pipeline(t5_input, max_new_tokens=max_target_length, num_beams=5, early_stopping=True)
    expanded_query = expanded_query_result[0]['generated_text']

    # 2. Topic Tagging (DistilBERT)
    bert_result = bert_pipeline(expanded_query)
    predicted_topic = bert_result[0]['label']
    confidence = bert_result[0]['score']

    return {
        "original_query": current_user_query_str,
        "expanded_query": expanded_query,
        "predicted_topic": predicted_topic,
        "confidence": confidence
    }

In [ ]:
history_1 = ""
query_1 = "who is pm of india"
pred_1 = get_full_prediction(history_1, query_1)
print(f"Pred 1: {pred_1}")

history_2 = "user: who is pm of india | bot: prime minister of India is narendra modi"
query_2 = "what are his duties"
pred_2 = get_full_prediction(history_2, query_2)
print(f"Pred 2: {pred_2}")

history_4 = "user: who is pm of india | bot: prime minister of India is narendra modi | user: what are his duties | bot: <answer for this>"
query_4 = "Give me minute, I am coming back"
pred_4 = get_full_prediction(history_4, query_4)
print(f"Pred 4: {pred_4}")

Pred 1: {'original_query': 'who is pm of india', 'expanded_query': 'Who is pm of India', 'predicted_topic': 'Polity & Governance - India', 'confidence': 0.463050901889801}
Pred 2: {'original_query': 'what are his duties', 'expanded_query': 'What are his duties as prime minister of India?', 'predicted_topic': 'Polity & Governance - India', 'confidence': 0.4973607659339905}
Pred 4: {'original_query': 'Give me minute, I am coming back', 'expanded_query': 'Give me minute, I am coming back,', 'predicted_topic': 'General Query - Meta', 'confidence': 0.7850630283355713}


In [ ]:
## Add dataset of UK, US, ...

In [ ]:
from huggingface_hub import HfApi, login

t5_repo_id = "metechmohit/final_t5_expansion_model"
bert_repo_id = "metechmohit/final_bert_topic_model"

api = HfApi()

# --- 1. UPLOAD T5 MODEL & TOKENIZER ---
print(f"--- Ensuring repo exists: {t5_repo_id} ---")
api.create_repo(
    repo_id=t5_repo_id,
    repo_type="model",
    exist_ok=True  # This is the correct way to handle "create_repo=True"
)

print(f"--- Uploading T5 Model Folder (ALL FILES) to {t5_repo_id} ---")
api.upload_folder(
    folder_path="./final_t5_expansion_model",
    repo_id=t5_repo_id,
    repo_type="model",
    commit_message="Upload all model and tokenizer files"
)
print("T5 folder upload complete.")


# --- 2. UPLOAD BERT MODEL & TOKENIZER ---
print(f"\n--- Ensuring repo exists: {bert_repo_id} ---")
api.create_repo(
    repo_id=bert_repo_id,
    repo_type="model",
    exist_ok=True  # This will not fail if the repo already exists
)

print(f"--- Uploading BERT Model Folder (ALL FILES) to {bert_repo_id} ---")
api.upload_folder(
    folder_path="./final_bert_topic_model",
    repo_id=bert_repo_id,
    repo_type="model",
    commit_message="Upload all model and tokenizer files"
)
print("BERT folder upload complete.")

--- Ensuring repo exists: metechmohit/final_t5_expansion_model ---
--- Uploading T5 Model Folder (ALL FILES) to metechmohit/final_t5_expansion_model ---


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n_model/training_args.bin: 100%|##########| 5.78kB / 5.78kB            

  ...ansion_model/spiece.model: 100%|##########|  792kB /  792kB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...c_model/training_args.bin: 100%|##########| 5.78kB / 5.78kB            

  ...c_model/model.safetensors:  16%|#5        | 41.9MB /  268MB            

BERT folder upload complete.
